# Notebook 1: Cleaner & Normalizer Test

**Objective**: Scrutinize header detection, cleaning, and normalization logic.

**Key questions**:
- Does auto header detection work for this file?
- Which columns are flagged as multi-valued?
- What bridge tables does normalization create?

In [ ]:


# def find_file(target_name):
#     """Searches likely paths for the test file."""
#     candidates = [
#         project_root / "data" / "raw" / target_name,
#         project_root / "data" / "synthetic" / target_name,
#         project_root / target_name,
#         current_dir / target_name
#     ]
#     for p in candidates:
#         if p.exists():
#             return p
#     return None

# # 4. Locate File
# TARGET_FILE = "professional_incident_tickets.csv"#"SLP_SEA_2025_Core.xlsx"
# file_path = find_file(TARGET_FILE)

# if file_path:
#     log(f"Found file at: {file_path}", "ok")
#     log(f"Project Root detected as: {project_root}")
# else:
#     log(f"File '{TARGET_FILE}' not found!", "err")
#     log(f"Please place it in: {project_root / 'data' / 'raw'}", "warn")
    
#     # Fallback: Search for ANY xlsx file in data/raw to help you debug
#     alt_files = list((project_root / "data" / "raw").glob("*.xlsx"))
#     if alt_files:
#         log(f"I found these other Excel files there:", "info")
#         for f in alt_files:
#             display(Markdown(f"- `{f.name}`"))
#     else:
#         log("data/raw/ directory is empty or missing.", "warn")

# # Set global variable for the rest of the notebook
# TEST_FILE_PATH = str(file_path) if file_path else None

In [1]:
# ── Setup & File Discovery ─────────────────────────────────────────────────────
import sys, os, glob, json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, HTML

# 1. Smart Path Resolution
# Notebooks usually run from their own folder (e.g. hybridtablerag/notebooks/)
current_dir = Path.cwd()
if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

# Ensure project root is on sys.path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 2. Imports
from hybridtablerag.core.cleaner import read_file, clean_dataframe
from hybridtablerag.core.profiler import profile_dataframe
from hybridtablerag.core.normalizer import Normalizer
from hybridtablerag.llm.factory import get_llm

# 3. Helpers
def log(msg, level="info"):
    colors = {"info": "#2563eb", "warn": "#b45309", "err": "#dc2626", "ok": "#16a34a"}
    display(HTML(f"<div style='font-family:monospace;font-size:.85rem;color:{colors.get(level,'#0f172a')}'>› {msg}</div>"))


In [2]:

#  Load File Using cleaner.py 

#FILE_PATH = r"c:\Users\Nikita.Dey\Documents\training py\HybridTableRag\data\raw\SLP_SEA_2025_Core.xlsx"
FILE_PATH = r"D:\Nikita\AI ML Engineer\s2s dynamics\HybridTableRag\data\synthetic\professional_incident_tickets.csv"

# For two-row header: pass [0, 1]
sheets = read_file(FILE_PATH, header_rows=[0, 1])

print(f"Loaded sheets: {list(sheets.keys())}")

# ── Step 2: Clean ──────────────────────────
from hybridtablerag.storage.store import DuckDBStore
DB_PATH = "data/hybridtablerag_test.duckdb"

store = DuckDBStore(db_path=DB_PATH)

# Drop everything explicitly
for t in store.list_tables():
    store.conn.execute(f'DROP TABLE "{t}"')

print("Clean DB ready")

cleaned_sheets = {}
for name, df in sheets.items():
    cleaned, log = clean_dataframe(df, log=[])
    cleaned_sheets[name] = cleaned
    
    
    display(Markdown(f"### Cleaned: `{name}`"))
    display(Markdown(f"**Columns**: `{list(cleaned.columns)}`"))
    display(cleaned.head(3))
    

Loaded sheets: ['sheet']
Clean DB ready


### Cleaned: `sheet`

**Columns**: `['ticket_ticket_id', 'ticket_created_date', 'ticket_resolved_date', 'ticket_priority', 'ticket_impacted_departments', 'customer_name', 'customer_email', 'resolution_affected_systems', 'resolution_actions_taken', 'resolution_description']`

,ticket_ticket_id,ticket_created_date,ticket_resolved_date,ticket_priority,ticket_impacted_departments,customer_name,customer_email,resolution_affected_systems,resolution_actions_taken,resolution_description
0,TCKT-1000,2024-05-03,2024-05-29,Medium,Finance;IT;Operations,Gabriella Davis,trevinomichele@example.net,"[{""system"": ""Payroll System"", ""impact"": ""Full""}]","[{""step"": ""Involve gun democratic several incl...",Of woman energy trade suddenly process you. Pe...
1,TCKT-1001,2025-03-31,2025-04-28,Low,HR;IT,Tiffany Howell,NaN,"[{""system"": ""Payroll System"", ""impact"": ""Parti...","[{""step"": ""True car site."", ""performed_by"": ""C...",Machine bar minute. Traditional us production.
2,TCKT-1002,2024-07-10,2024-08-01,Medium,Sales;Operations,Brandon Maxwell,joescott@example.com,"[{""system"": ""Database"", ""impact"": ""Full""}, {""s...","[{""step"": ""Read forget issue Democrat kitchen ...",Brother build scene interview. Later idea assu...


In [3]:
# ── Diagnostic: Verify Multi-Value Detection ─────────────────
from hybridtablerag.core.profiler import profile_dataframe

for name, df in cleaned_sheets.items():
    display(Markdown(f"### Profile: `{name}`"))
    
    profile = profile_dataframe(df)
    
    # Show only multi-valued or JSON columns
    interesting = {
        col: stats for col, stats in profile["columns"].items()
        if stats["is_multi_valued"] or 
           any(s in str(stats["sample_values"][:2]).lower() for s in [';', '[', '{', '|', ','])
    }
    
    if interesting:
        summary_rows = []
        for col, stats in interesting.items():
            summary_rows.append({
                "column": col,
                "dtype": stats["dtype"],
                "multi?": stats["is_multi_valued"],
                "type": stats["multi_val_type"],
                "sample": " | ".join(str(v)[:50] for v in stats["sample_values"][:2]),
            })
        display(pd.DataFrame(summary_rows).set_index("column"))
    else:
        display(Markdown("No multi-valued columns detected — check dtype/heuristics"))

### Profile: `sheet`

,dtype,multi?,type,sample
column,,,,
ticket_ticket_id,str,False,NaN,TCKT-1000 | TCKT-1001
ticket_created_date,str,False,NaN,2024-05-03 | 2025-03-31
ticket_resolved_date,str,False,NaN,2024-05-29 | 2025-04-28
ticket_priority,str,False,NaN,Medium | Low
ticket_impacted_departments,str,True,semicolon,Finance;IT;Operations | HR;IT
customer_name,str,False,NaN,Gabriella Davis | Tiffany Howell
customer_email,str,False,NaN,trevinomichele@example.net | joescott@example.com
resolution_affected_systems,str,True,comma,"[{""system"": ""Payroll System"", ""impact"": ""Full""..."
resolution_actions_taken,str,True,comma,"[{""step"": ""Involve gun democratic several incl..."


In [4]:
# ── Normalization Test ────────────────────────────────────────────────
from hybridtablerag.core.normalizer import NormalizationPlan


print("Testing Normalizer with LLM…")

llm = get_llm()  # Reads from .env
normalizer = Normalizer(llm=llm)

norm_results = {}
for name, df in cleaned_sheets.items():
    print(f"Normalizing `{name}`…")
    
    try:
        plan: NormalizationPlan = normalizer.normalize(
            df, 
            table_hint=name.replace(" ", "_").lower(),
            profile_hints=profile["columns"],
            log=[]
        )
        
        norm_results[name] = plan
        
        # Show results
        display(Markdown(f"#### Normalization Plan: `{name}`"))
        display(Markdown(f"**Main table**: `{plan.main_table_name}` ({len(plan.main_df)} rows)"))
        
        if plan.bridge_tables:
            display(Markdown("**Bridge tables created**:"))
            for bt in plan.bridge_tables:
                display(Markdown(
                    f"- `{bt.name}`: {len(bt.df)} rows | "
                    f"cols: {list(bt.df.columns)} | "
                    f"from: `{bt.source_col}` (sep: `{bt.separator}`)"
                ))
        else:
            display(Markdown("No bridge tables needed"))
        
        if plan.relationships:
            display(Markdown("**Relationships**:"))
            for r in plan.relationships:
                display(Markdown(
                    f"- `{r['from_table']}.{r['from_column']}` → "
                    f"`{r['to_table']}.{r['to_column']}` ({r['type']})"
                ))
                
    except Exception as e:
        print(f"Normalization failed for `{name}`: {e}", "err")
        import traceback
        display(Markdown(f"```\n{traceback.format_exc()}\n```"))

Testing Normalizer with LLM…
Normalizing `sheet`…


#### Normalization Plan: `sheet`

**Main table**: `sheet` (1000 rows)

**Bridge tables created**:

- `sheet_department`: 1996 rows | cols: ['ticket_ticket_id', 'department'] | from: `ticket_impacted_departments` (sep: `;`)

...

- `sheet_resolution_action`: 2032 rows | cols: ['ticket_ticket_id', 'step', 'performed_by'] | from: `resolution_actions_taken` (sep: `json`)

**Relationships**:

- `sheet_department.ticket_ticket_id` → `sheet.ticket_ticket_id` (many_to_one)

- `sheet_system.ticket_ticket_id` → `sheet.ticket_ticket_id` (many_to_one)

- `sheet_resolution_action.ticket_ticket_id` → `sheet.ticket_ticket_id` (many_to_one)

In [5]:
# ── Direct Registration to DuckDB (Disk Storage) ───────────────
import os
from pathlib import Path
from hybridtablerag.storage.store import DuckDBStore
from hybridtablerag.storage.vectors import VectorStore, get_embedding_provider
import pandas as pd
from IPython.display import display, Markdown

# 1. Use a real file path (persists across notebook restarts)
DB_PATH = "data/hybridtablerag_test.duckdb"

# Delete old test DB to start clean
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print(f"Deleted old {DB_PATH}")

# 2. Initialize store
store = DuckDBStore(db_path=DB_PATH)
print(f"Connected to: {store.db_path}")

# 3. Register NormalizationPlan directly (NO CSV!)
bts_log = []
store.register_normalization_plan(plan, bts_log)

print("Registered tables:", store.list_tables())
display(Markdown("### Registered Tables"))
display(pd.DataFrame({"Table": store.list_tables()}))

# 4. Quick verification
display(Markdown(f"### Main Table Preview: `{plan.main_table_name}`"))
display(store.execute_query(f"SELECT * FROM {plan.main_table_name} LIMIT 3"))

#all tables
tables = store.list_tables()

for t in tables:
    print(f"\nChecking table: {t}")
    df = store.conn.execute(f"SELECT * FROM {t} LIMIT 3").fetchdf()
    display(df)




Connected to: d:\Nikita\AI ML Engineer\s2s dynamics\HybridTableRag\data/hybridtablerag_test.duckdb
Registered tables: ['sheet', 'sheet_department', 'sheet_resolution_action', 'sheet_system']


### Registered Tables

,Table
0,sheet
1,sheet_department
2,sheet_resolution_action
3,sheet_system


### Main Table Preview: `sheet`

,ticket_ticket_id,ticket_created_date,ticket_resolved_date,ticket_priority,customer_name,customer_email,resolution_description
0,TCKT-1000,2024-05-03,2024-05-29,Medium,Gabriella Davis,trevinomichele@example.net,Of woman energy trade suddenly process you. Pe...
1,TCKT-1001,2025-03-31,2025-04-28,Low,Tiffany Howell,NaN,Machine bar minute. Traditional us production.
2,TCKT-1002,2024-07-10,2024-08-01,Medium,Brandon Maxwell,joescott@example.com,Brother build scene interview. Later idea assu...


...

,ticket_ticket_id,department
0,TCKT-1000,Finance
1,TCKT-1000,IT
2,TCKT-1000,Operations



Checking table: sheet_resolution_action


,ticket_ticket_id,step,performed_by
0,TCKT-1000,Involve gun democratic several including wish.,Victoria Brown
1,TCKT-1000,Concern hair project country onto wrong.,Cassandra Jones
2,TCKT-1001,True car site.,Connie Johnson



Checking table: sheet_system


,ticket_ticket_id,system,impact
0,TCKT-1000,Payroll System,Full
1,TCKT-1001,Payroll System,Partial
2,TCKT-1002,Database,Full


In [6]:
# ── Embed ALL Tables ───────────────
from hybridtablerag.storage.store import DuckDBStore
from hybridtablerag.storage.vectors import VectorStore, get_embedding_provider
import re

DB_PATH = "data/hybridtablerag_test.duckdb"

store = DuckDBStore(db_path=DB_PATH)



provider = get_embedding_provider()
vector_store = VectorStore(store.conn, provider)
vector_store.setup()

print(f"VectorStore ready (dim: {provider.dimension})")

bts_log = []

# Collect all tables
all_tables = {plan.main_table_name: plan.main_df}
for bt in plan.bridge_tables:
    all_tables[bt.name] = bt.df


def is_semantic_column(store, table_name, col):
    # sample values
    rows = store.conn.execute(
        f'SELECT "{col}" FROM "{table_name}" WHERE "{col}" IS NOT NULL LIMIT 50'
    ).fetchall()

    values = [str(r[0]) for r in rows if r[0] is not None]
    if not values:
        return False

    # --- Pattern filters (lightweight) ---
    email_pattern = re.compile(r"\S+@\S+\.\S+")
    date_pattern = re.compile(r"^\d{4}-\d{2}-\d{2}")

    if all(email_pattern.match(v) for v in values):
        return False

    if all(date_pattern.match(v) for v in values):
        return False

    # --- Compute signals ---
    total_chars = sum(len(v) for v in values)
    alpha_chars = sum(sum(c.isalpha() for c in v) for v in values)

    alpha_ratio = alpha_chars / total_chars if total_chars else 0

    avg_token_len = sum(len(v.split()) for v in values) / len(values)

    distinct = store.conn.execute(
        f'SELECT COUNT(DISTINCT "{col}") FROM "{table_name}"'
    ).fetchone()[0]

    total = store.conn.execute(
        f'SELECT COUNT(*) FROM "{table_name}"'
    ).fetchone()[0]

    cardinality_ratio = distinct / total if total else 0

    # --- Decision logic ---
    if alpha_ratio > 0.5 and avg_token_len >= 1:
        return True

    # allow low-cardinality categorical columns
    if distinct < 50 and alpha_ratio > 0.6:
        return True

    return False


for table_name in all_tables.keys():
    print(f"\nProcessing table: {table_name}")
    
    schema = store.get_table_schema(table_name)

    # ── Detect PK safely ───────────────────────────────────────────
    pk_column = None
    for col_info in schema:
        col = col_info["column_name"]
        if col.endswith("_id"):
            pk_column = col
            break

    if pk_column is None:
        print(f"Skipping {table_name}: No PK found")
        continue

    # ── Detect text columns ────────────────────────────────────────
    text_cols = []
    
    text_cols = []

    for col_info in schema:
        col = col_info["column_name"]
        dtype = col_info["data_type"]

        is_text = dtype in ("VARCHAR", "TEXT")
        is_pk = col == pk_column

        if not is_text or is_pk:
            continue

        if is_semantic_column(store, table_name, col):
            text_cols.append(col)


    print(f"Embedding columns: {text_cols}")

    # ── Embed using ALL text columns ───────────────────────────────
    vector_store.embed_table(
        table_name=table_name,
        text_columns=text_cols,   
        pk_column=pk_column,      
        bts_log=bts_log
    )

print("\nVector embedding complete.")
print("\nLogs:")
for log in bts_log:
    print("-", log)



d:\Nikita\AI ML Engineer\s2s dynamics\HybridTableRag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5019.44it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


VectorStore ready (dim: 384)

Processing table: sheet
Embedding columns: ['ticket_priority', 'customer_name', 'resolution_description']

Processing table: sheet_department
Embedding columns: ['department']

Processing table: sheet_system
Embedding columns: ['system', 'impact']

Processing table: sheet_resolution_action
Embedding columns: ['step', 'performed_by']

Vector embedding complete.

Logs:
- sheet: Embedded 1000 rows
- sheet_department: Embedded 1996 rows
- sheet_system: Embedded 1981 rows
- sheet_resolution_action: Embedded 2032 rows


In [7]:
vector_store.search(
    "payroll system failure",
    "sheet"
)

,ticket_ticket_id,ticket_created_date,ticket_resolved_date,ticket_priority,customer_name,customer_email,resolution_description,similarity_score
0,TCKT-1409,2026-01-08,2026-01-14,Critical,Lisa Baker,vaughnjulie@example.com,Gas fill this see figure. As central low walk....,1.217734
1,TCKT-1234,2025-07-20,2025-08-10,Critical,Crystal Hale,alee@example.net,Loss research office similar table trouble act...,1.218205
2,TCKT-1619,2024-08-28,2024-09-13,Medium,Jeffrey Cabrera,patrick84@example.com,None free job. Base final everything lose. Att...,1.218271
3,TCKT-1534,2025-04-04,2025-10-04,Critical,Melinda Murphy,sarahcampbell@example.com,Country machine both surface add find her seat...,1.236376
4,TCKT-1978,2024-12-27,2025-01-13,Low,Samantha Reese,christopher93@example.com,Case catch suddenly heavy.,1.244425
5,TCKT-1877,2025-12-23,2026-01-21,Critical,Tiffany Wade,christine98@example.org,Stay full lot again least collection stand. El...,1.245596
6,TCKT-1674,2025-03-21,2025-03-26,Medium,Bryan Johnson,pwilson@example.org,Pick through customer too worker short attack....,1.247825
7,TCKT-1713,2025-12-13,2025-12-20,Low,Nicholas Moore,jerryboyd@example.com,Single account lose though. Section manager ho...,1.247866
8,TCKT-1725,2025-12-08,2025-12-22,Critical,Elizabeth Martinez,asalazar@example.org,Myself court support visit hospital mean. Requ...,1.251143
9,TCKT-1768,2026-02-23,2026-03-03,Medium,Kelly Callahan,carlhamilton@example.org,Staff new alone consumer early close. Watch ac...,1.252009


In [ ]:
# llm=get_llm()
# print(llm.generate("Say hello"))

Hello! How can I help you today?


In [8]:
from hybridtablerag.reasoning.intent import IntentClassifier
from hybridtablerag.storage.schema import build_multi_table_schema_context, format_schema_for_prompt

llm = get_llm()  

intent = IntentClassifier(llm)
bts_log=[]
schema_ctx = build_multi_table_schema_context(
    store.conn,
    list(all_tables.keys()),
    relationships=[],
    bts_log=bts_log
)

schema_summary = format_schema_for_prompt(schema_ctx)

query = "show ticket count by status"

plan = intent.classify(
    user_query=query,
    schema_summary=schema_summary,
    available_tables=list(all_tables.keys())
)

print(plan)

QueryPlan(needs_sql=True, needs_python=False, needs_semantic=False, is_conversational=False, relevant_tables=['sheet'], needs_aggregation=True, needs_join=False, needs_ranking=False, python_mode='table', visualization_reason='A bar chart showing counts per status would clearly compare Open vs Resolved tickets.', semantic_column_hint='', semantic_query="SELECT CASE WHEN ticket_resolved_date IS NOT NULL THEN 'Resolved' ELSE 'Open' END AS status, COUNT(*) AS cnt FROM sheet GROUP BY status", reasoning='We derive a ticket status from the resolution date and aggregate counts to show the distribution by status.')


In [9]:
from hybridtablerag.reasoning.sql import LLMSQLGenerator

sql_gen = LLMSQLGenerator(llm)

sql = sql_gen.generate_sql(
    user_query="show ticket count by status",
    schema_metadata=schema_ctx,
    relationships=[],
    reasoning=True,
    plan=plan
)

print(sql)

QueryPlan(needs_sql=True, needs_python=False, needs_semantic=False, is_conversational=False, relevant_tables=['sheet'], needs_aggregation=True, needs_join=False, needs_ranking=False, python_mode='table', visualization_reason='A bar chart showing counts per status would clearly compare Open vs Resolved tickets.', semantic_column_hint='', semantic_query="SELECT CASE WHEN ticket_resolved_date IS NOT NULL THEN 'Resolved' ELSE 'Open' END AS status, COUNT(*) AS cnt FROM sheet GROUP BY status", reasoning='We derive a ticket status from the resolution date and aggregate counts to show the distribution by status.')
{'sql_query': "SELECT\n  CASE\n    WHEN CAST(ticket_resolved_date AS DATE) <= CURRENT_DATE THEN 'Resolved'\n    ELSE 'Open'\n  END AS status,\n  COUNT(*) AS ticket_count\nFROM sheet\nGROUP BY 1;", 'reasoning': "We derive a ticket status by comparing each ticket's resolution date to the current date. If the resolution date is on or before today, the status is 'Resolved'; otherwise, it

In [11]:
from hybridtablerag.reasoning.orchestrator import QueryOrchestrator

orchestrator = QueryOrchestrator(
    llm=llm,
    store=store,
    context_store=None,
    sql_generator=sql_gen,
    table_names=list(all_tables.keys()),
    relationships=[],
    vector_store=vector_store,
    default_table=list(all_tables.keys())[0],
)

result = orchestrator.run(
    user_query="show ticket count by status",
    session_id="test_session",
    debug_mode=True
)

print("Intent:", result.intent)
print("SQL:", result.sql)
print("Error:", result.error)

if result.dataframe is not None:
    display(result.dataframe.head())

print("\n--- BTS LOG ---")
for log in result.bts_log:
    print(log)

QueryPlan(needs_sql=True, needs_python=False, needs_semantic=False, is_conversational=False, relevant_tables=['sheet'], needs_aggregation=True, needs_join=False, needs_ranking=False, python_mode='table', visualization_reason='Bar chart showing the count of tickets by status (Open vs Resolved) to compare distribution at a glance.', semantic_column_hint='', semantic_query='', reasoning='We can derive status from resolution_date in sheet (Open if null, Resolved if present), then aggregate counts by that status via a single SQL GROUP BY.')
Intent: sql
SQL: SELECT
  CASE WHEN ticket_resolved_date IS NULL THEN 'Open' ELSE 'Resolved' END AS status,
  COUNT(*) AS ticket_count
FROM sheet
GROUP BY CASE WHEN ticket_resolved_date IS NULL THEN 'Open' ELSE 'Resolved' END
ORDER BY status;
Error: None


,status,ticket_count
0,Resolved,1000



--- BTS LOG ---
 Building schema context…
Failed to fetch stats for _embedding: BinderException
Built schema context for table: sheet
Built schema context for table: sheet_department
Built schema context for table: sheet_system
Failed to fetch stats for _embedding: BinderException
Built schema context for table: sheet_resolution_action
Plan: sql=True python=False semantic=False conv=False mode=table
   Reasoning: We can derive status from resolution_date in sheet (Open if null, Resolved if present), then aggregate counts by that status via a single SQL GROUP BY.
Using tables: ['sheet']
SQL attempt 1/3
Generated SQL:
SELECT
  CASE WHEN ticket_resolved_date IS NULL THEN 'Open' ELSE 'Resolved' END AS status,
  COUNT(*) AS ticket_count
FROM sheet
GROUP BY CASE WHEN ticket_resolved_date IS NULL THEN 'Open' ELSE 'Resolved' END
ORDER BY status;
SQL returned 1 rows × 2 cols


In [12]:
result = orchestrator.run(
    user_query="show distribution of tickets by priority with a chart",
    session_id="test_session",
    debug_mode=True
)

if result.chart:
    result.chart.show()

display(result)

QueryPlan(needs_sql=True, needs_python=True, needs_semantic=False, is_conversational=False, relevant_tables=['sheet'], needs_aggregation=True, needs_join=False, needs_ranking=False, python_mode='table', visualization_reason='Bar chart showing distribution of tickets by priority to compare how many tickets fall into each priority level (Low/Medium/High/Critical).', semantic_column_hint='', semantic_query='', reasoning='Aggregate counts by ticket_priority from sheet and plot a bar chart to visualize the distribution across priority levels.')


QueryResult(intent='sql+python', user_query='show distribution of tickets by priority with a chart', session_id='test_session', sql='SELECT ticket_priority AS priority, COUNT(*) AS priority_count\nFROM sheet\nGROUP BY ticket_priority\nORDER BY priority_count DESC;', dataframe=   priority  priority_count
0    Medium             273
1      High             254
2  Critical             249
3       Low             224, reasoning=None, python_code='', python_dataframe=   priority  priority_count
0    Medium             273
1      High             254
2  Critical             249
3       Low             224, chart=None, vector_results=None, vector_query=None, llm_answer=None, context_used=None, error=None, python_error=None, vector_error=None, debug_info={'schema_tables': ['sheet'], 'plan': {'needs_sql': True, 'needs_python': True, 'needs_semantic': False, 'python_mode': 'table', 'relevant_tables': ['sheet'], 'reasoning': 'Aggregate counts by ticket_priority from sheet and plot a bar chart to 

In [13]:
result = orchestrator.run(
    user_query="find tickets about payroll issues",
    session_id="test_session",
    debug_mode=True
)

if result.dataframe is not None:
    display(result.dataframe.head())

if result.vector_results is not None:
    display(result.vector_results.head())

QueryPlan(needs_sql=True, needs_python=False, needs_semantic=False, is_conversational=False, relevant_tables=['sheet', 'sheet_affected_system'], needs_aggregation=False, needs_join=True, needs_ranking=False, python_mode='table', visualization_reason='', semantic_column_hint='', semantic_query='', reasoning='Plan is to SQL-filter tickets by payroll-related deployment in the Payroll System and return their details (no aggregation), using a join between sheet and sheet_affected_system for precise matching.')


,ticket_id,ticket_created_date,ticket_resolved_date,ticket_priority,customer_name,customer_email,resolution_description,affected_system,impact
0,TCKT-1587,2024-01-05,2024-05-15,High,Brian Daniels,troy40@example.com,Energy recent position bank its amount. Pressu...,Payroll System,Full
1,TCKT-1944,2024-01-08,2024-08-21,Critical,Randy Patterson,paulrobert@example.net,Value call final difficult open at. Through sa...,Payroll System,Partial
2,TCKT-1669,2024-01-10,2024-10-13,High,Mary Gonzalez,campbellsusan@example.com,Present will live amount page popular poor. Ma...,Payroll System,Partial
3,TCKT-1459,2024-01-12,2024-12-12,High,Derrick Hoover,ujohnson@example.org,Here image than them arrive information work. ...,Payroll System,Partial
4,TCKT-1552,2024-02-03,2024-03-15,High,Jennifer Winters,mindypeters@example.org,Able mind order father. Side cup despite soon ...,Payroll System,Partial
